# From “fake news model” to a **product**: unseen articles, automation, and honest limits

**Audience:** instructors and students building a **generic triage** layer for many articles (your own site, partner feeds, or licensed content).

---

## 1. What we automate vs. what we do not

| Automated well | Not automated by a single classifier |
|----------------|--------------------------------------|
| **Ranking** unseen articles by risk so humans review the top of the queue | **Ground truth** (“this claim is false in the real world”) |
| **Consistent first pass** using the same model version | **Legal / ethical** approval to publish |
| **Throughput** at scale (API + batch jobs) | **AI vs human author** (different detectors / policies) |

**ML helps** newsrooms the same way spam filters help email: **prioritization**, not perfect truth.

---

## 2. Unseen data in production

1. **Train** on historical labeled data (domain matters).
2. **Freeze** artifacts (`logreg_tfidf.joblib`, Keras `model.keras`, or a Hugging Face export).
3. **Serve** a small API: new text in → score + optional explanation out.
4. **Monitor** drift (distribution of scores, sudden accuracy drop on a small human-audited sample).
5. **Retrain** when labels or the world change (election cycles, new meme formats, LLM slop).

This repo’s **`/api/analyze`** and **`/api/analyze-url`** illustrate steps 3–4 for demos (URL fetch only where you have rights).

---

## 3. Model zoo (classical → deep → transformers → LLM)

| Approach | When to use | Caveat for “fake news” |
|----------|-------------|-------------------------|
| **TF–IDF + linear** | Baseline, interpretable, fast | Picks up **dataset shortcuts** (site names, boilerplate) |
| **LSTM / CNN / small Transformer (Keras)** | More capacity, still trainable on one GPU | Needs **more data**; explainability needs extra tooling |
| **Fine-tuned BERT / RoBERTa (PyTorch or TF)** | Best generic text classification transfer | **Not** calibrated “probability of lying” without Platt / isotonic calibration on *your* data |
| **LLM + prompts / RAG** | Copilot for editors, draft questions, retrieve evidence | **Hallucinations**; treat as assistant; log prompts and versions |

**LLM-generated articles:** detectors exist (separate training corpora). Do not confuse “looks like our ‘reliable’ class” with “written by a person.”

---

## 4. Accuracy on “larger data”

- **More data** usually helps **until** the task definition blurs (mixing satire, opinion, and reportage).
- **Active learning:** humans label the items the model is **uncertain** about—best ROI for labeling budget.
- **Metrics:** accuracy alone is misleading under **class imbalance**; track **precision/recall** at the threshold you use for routing, and **calibration** if you show percentages to users.

Run locally after training: open `artifacts/metrics.json` or use notebook `02_metrics_and_overfitting.ipynb`.

---

## 5. Hands-on (run after `pip install -e ".[dev]"` and training)

Execute the next cells from the **repository root** (or fix `PROJECT_ROOT`).

In [ ]:
from pathlib import Path
import sys
import json

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.service.predictor import build_api_response, product_framing

product_framing()

In [ ]:
# Simulate one unseen document (replace with your own headline + body)
from src.data.preprocess import combine_title_body

title = "Mayor announces flood relief fund after weekend storms"
body = (
    "City officials said applications will open Monday at the civic center. "
    "The state emergency agency confirmed it will audit disbursements quarterly."
)
text = combine_title_body(title, body)
out = build_api_response(text, "classical", teacher_mode=False)
print("Score (toward review):", out["score_toward_review_0_to_1"])
print("Summary:", out["user_summary"]["headline"])
print("Why (short):", out["executive_why"][:500], "...")

In [ ]:
# Optional: batch many unseen rows from a CSV you own (columns: title, body or text)
# import pandas as pd
# df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "test.csv")
# from src.service.predictor import predict_proba_fake
# scores = [predict_proba_fake(t, "classical") for t in df["text"].head(200)]
# sum(s > 0.5 for s in scores if s is not None), len(scores)

## 6. Student exercises

1. **Threshold policy:** Pick two thresholds and write who reviews what (editorial policy doc).
2. **Failure modes:** Find a **true** article that scores high “review” and a **false** one that scores low; explain using `explain_classical_for_text`.
3. **Extension:** Fine-tune `distilbert-base-uncased` in PyTorch on `train.csv` and compare ROC-AUC to the linear baseline on `test.csv`.
4. **LLM:** Sketch a RAG flow (trusted corpus + retrieval + prompt) **without** claiming it replaces your classifier.

Further reading for the class: **`PRODUCT_AUTOMATION.md`** in the repo root.